# Text Classification: BiLSTM NLP Model & Experiment Tracking

Notebook ini berisi langkah-langkah untuk:
1. **Deep Learning Modelling**: Melatih model **Bidirectional LSTM** dengan Keras Embedding layer.
2. **Hyperparameter Tweaking**: Menjalankan 4 konfigurasi eksperimen berbeda yang di-log ke MLflow.
3. **Model Terbaik**: Membandingkan hasil eksperimen dan menampilkan model terbaik berdasarkan F1-Score.

Data yang digunakan adalah data hasil pembersihan dari tahap sebelumnya.
Untuk mengubah konfigurasi eksperimen, edit nilai di `EXPERIMENTS_CONFIG`.

In [ ]:
import pandas as pd
import numpy as np
import time
import os
import warnings
import urllib3

urllib3.disable_warnings()
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)

import mlflow
import mlflow.tensorflow

print("TensorFlow version:", tf.__version__)
print("MLflow version:", mlflow.__version__)

### 1. Load Dataset & Preprocessing (Shared Across All Experiments)

In [ ]:
df = pd.read_csv('results/data/dataset_cleaned.csv')
df = df.dropna(subset=['cleaned_text', 'sentiment'])

print("Total data:", len(df))
print("\nDistribusi Kelas:")
print(df['sentiment'].value_counts())

# Label Encoding
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])
num_classes = len(le.classes_)
print(f"\nJumlah kelas: {num_classes}")
print("Kelas:", le.classes_.tolist())

In [ ]:
# Train-Test Split (dilakukan SEKALI, dipakai semua eksperimen)
X_raw = df['cleaned_text'].astype(str).values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train_raw)}, Test: {len(X_test_raw)}")

In [ ]:
# Class Weights (menggantikan SMOTE — lebih tepat untuk deep learning)
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes.tolist(), weights.tolist()))

print("Class Weights:")
for idx, name in enumerate(le.classes_):
    print(f"  {name}: {class_weight_dict[idx]:.4f}")

### 2. Konfigurasi Eksperimen (PARAMS)

Edit nilai di `EXPERIMENTS_CONFIG` untuk mencoba hyperparameter yang berbeda.
Setiap entry akan dijalankan sebagai satu MLflow run terpisah.

| Parameter | Keterangan |
|---|---|
| `vocab_size` | Jumlah kata teratas yang dipakai sebagai fitur |
| `max_len` | Panjang maksimum urutan token (padding/truncate) |
| `embedding_dim` | Dimensi vektor embedding per token |
| `lstm_units` | Jumlah unit LSTM (×2 karena Bidirectional) |
| `dropout_rate` | Kekuatan regularisasi dropout (0.0–1.0) |
| `learning_rate` | Learning rate optimizer Adam |
| `batch_size` | Jumlah sampel per batch training |
| `epochs` | Maksimum epoch (EarlyStopping bisa berhenti lebih awal) |

In [ ]:
EXPERIMENTS_CONFIG = [
    {
        "run_name": "BiLSTM_Baseline",
        "PARAMS": {
            "vocab_size":    10000,
            "max_len":          30,
            "embedding_dim":    64,
            "lstm_units":       64,
            "dropout_rate":    0.3,
            "learning_rate": 0.001,
            "batch_size":       64,
            "epochs":           20,
        }
    },
    {
        "run_name": "BiLSTM_LargerVocab",
        "PARAMS": {
            "vocab_size":    20000,
            "max_len":          30,
            "embedding_dim":   128,
            "lstm_units":      128,
            "dropout_rate":    0.3,
            "learning_rate": 0.001,
            "batch_size":       64,
            "epochs":           20,
        }
    },
    {
        "run_name": "BiLSTM_DeepDropout",
        "PARAMS": {
            "vocab_size":    20000,
            "max_len":          30,
            "embedding_dim":   128,
            "lstm_units":      128,
            "dropout_rate":    0.5,
            "learning_rate": 0.0005,
            "batch_size":       32,
            "epochs":           25,
        }
    },
    {
        "run_name": "BiLSTM_StackedLSTM",
        "PARAMS": {
            "vocab_size":    20000,
            "max_len":          30,
            "embedding_dim":   128,
            "lstm_units":      128,
            "dropout_rate":    0.4,
            "learning_rate": 0.001,
            "batch_size":       64,
            "epochs":           20,
        }
    },
]

### 3. Helper Functions: Tokenizer, Sequences, dan Model Builder

In [ ]:
def build_tokenizer_and_sequences(X_train_raw, X_test_raw, vocab_size, max_len):
    """Fit tokenizer on train only, then transform both splits."""
    tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train_raw)

    X_train_pad = pad_sequences(
        tokenizer.texts_to_sequences(X_train_raw),
        maxlen=max_len, padding='post', truncating='post'
    )
    X_test_pad = pad_sequences(
        tokenizer.texts_to_sequences(X_test_raw),
        maxlen=max_len, padding='post', truncating='post'
    )
    return tokenizer, X_train_pad, X_test_pad


def build_bilstm_model(vocab_size, max_len, embedding_dim, lstm_units,
                       dropout_rate, learning_rate, num_classes, stacked=False):
    """Build a Bidirectional LSTM model. stacked=True adds a second BiLSTM layer."""
    model = Sequential()
    # vocab_size + 1: Tokenizer indices run 0 (pad) to vocab_size inclusive
    model.add(Embedding(input_dim=vocab_size + 1,
                        output_dim=embedding_dim,
                        input_length=max_len))
    # SpatialDropout1D drops entire embedding dimensions — better than Dropout for sequences
    model.add(SpatialDropout1D(dropout_rate))

    if stacked:
        model.add(Bidirectional(LSTM(lstm_units, return_sequences=True, dropout=dropout_rate)))
        model.add(Bidirectional(LSTM(lstm_units // 2, dropout=dropout_rate)))
    else:
        model.add(Bidirectional(LSTM(lstm_units, dropout=dropout_rate)))

    model.add(Dense(64, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


print("Helper functions siap.")

### 4. Jalankan Eksperimen & Log ke MLflow

In [ ]:
mlflow.set_experiment("Emotion_Classification_Experiment")

experiment_logs = []

for config in EXPERIMENTS_CONFIG:
    run_name = config["run_name"]
    P = config["PARAMS"]
    is_stacked = (run_name == "BiLSTM_StackedLSTM")

    print(f"\n{'='*60}")
    print(f"Menjalankan: {run_name}")
    print(f"PARAMS: {P}")
    print(f"{'='*60}")

    # Bangun tokenizer & sequence baru per config (vocab_size bisa berbeda)
    _, X_train_pad, X_test_pad = build_tokenizer_and_sequences(
        X_train_raw, X_test_raw,
        vocab_size=P['vocab_size'],
        max_len=P['max_len']
    )

    # Bangun model
    model = build_bilstm_model(
        vocab_size=P['vocab_size'],
        max_len=P['max_len'],
        embedding_dim=P['embedding_dim'],
        lstm_units=P['lstm_units'],
        dropout_rate=P['dropout_rate'],
        learning_rate=P['learning_rate'],
        num_classes=num_classes,
        stacked=is_stacked
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=0
    )

    with mlflow.start_run(run_name=run_name):
        # Log semua hyperparameter
        mlflow.log_param("model_type", "BiLSTM")
        mlflow.log_param("stacked", is_stacked)
        for key, val in P.items():
            mlflow.log_param(key, val)

        # Training
        start_time = time.time()
        history = model.fit(
            X_train_pad, y_train,
            epochs=P['epochs'],
            batch_size=P['batch_size'],
            validation_split=0.1,
            class_weight=class_weight_dict,
            callbacks=[early_stop],
            verbose=1
        )
        training_time = time.time() - start_time
        epochs_ran = len(history.history['loss'])

        # Prediksi
        y_pred = np.argmax(model.predict(X_test_pad, verbose=0), axis=1)

        # Metrik
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        # Log metrik ke MLflow
        mlflow.log_metric("training_time", round(training_time, 4))
        mlflow.log_metric("accuracy",      round(acc,  4))
        mlflow.log_metric("precision",     round(prec, 4))
        mlflow.log_metric("recall",        round(rec,  4))
        mlflow.log_metric("f1_score",      round(f1,   4))

        # Log model
        mlflow.tensorflow.log_model(model, name="bilstm_model")

        print(f"\n{run_name} selesai.")
        print(f"  Epochs: {epochs_ran}/{P['epochs']} | F1-Score: {f1:.4f} | Accuracy: {acc:.4f} | Time: {training_time:.1f}s")

        experiment_logs.append({
            'Run Name':          run_name,
            'vocab_size':        P['vocab_size'],
            'max_len':           P['max_len'],
            'embedding_dim':     P['embedding_dim'],
            'lstm_units':        P['lstm_units'],
            'dropout_rate':      P['dropout_rate'],
            'learning_rate':     P['learning_rate'],
            'batch_size':        P['batch_size'],
            'epochs_ran':        epochs_ran,
            'Training Time (s)': round(training_time, 2),
            'Accuracy':          round(acc,  4),
            'Precision':         round(prec, 4),
            'Recall':            round(rec,  4),
            'F1-Score':          round(f1,   4),
        })

print("\nSemua eksperimen selesai.")

### 5. Evaluasi Eksperimen & Model Terbaik

In [ ]:
# Tampilkan tabel perbandingan semua eksperimen
df_logs = pd.DataFrame(experiment_logs)
df_logs = df_logs.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("=== LOG SEMUA EKSPERIMEN ===")
display(df_logs)

# Simpan log ke CSV
os.makedirs('results', exist_ok=True)
df_logs.to_csv('results/experiment_logs.csv', index=False)
print("\nLog eksperimen disimpan ke 'results/experiment_logs.csv'")

In [ ]:
# Model Terbaik — ringkasan
best_row = df_logs.iloc[0]

print("=" * 60)
print("MODEL TERBAIK")
print("=" * 60)
print(f"Run Name:         {best_row['Run Name']}")
print(f"F1-Score:         {best_row['F1-Score']}")
print(f"Accuracy:         {best_row['Accuracy']}")
print(f"Precision:        {best_row['Precision']}")
print(f"Recall:           {best_row['Recall']}")
print(f"Training Time:    {best_row['Training Time (s)']}s")
print(f"Epochs (actual):  {best_row['epochs_ran']}")
print("Hyperparameters:")
for col in ['vocab_size', 'max_len', 'embedding_dim', 'lstm_units',
            'dropout_rate', 'learning_rate', 'batch_size']:
    print(f"  {col}: {best_row[col]}")
print("=" * 60)

# Re-train model terbaik untuk classification report
best_run_name = best_row['Run Name']
best_config = next(c for c in EXPERIMENTS_CONFIG if c['run_name'] == best_run_name)
P_best = best_config['PARAMS']
is_stacked_best = (best_run_name == "BiLSTM_StackedLSTM")

_, X_train_best, X_test_best = build_tokenizer_and_sequences(
    X_train_raw, X_test_raw,
    vocab_size=P_best['vocab_size'],
    max_len=P_best['max_len']
)

best_model = build_bilstm_model(
    vocab_size=P_best['vocab_size'],
    max_len=P_best['max_len'],
    embedding_dim=P_best['embedding_dim'],
    lstm_units=P_best['lstm_units'],
    dropout_rate=P_best['dropout_rate'],
    learning_rate=P_best['learning_rate'],
    num_classes=num_classes,
    stacked=is_stacked_best
)

best_model.fit(
    X_train_best, y_train,
    epochs=P_best['epochs'],
    batch_size=P_best['batch_size'],
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=0)],
    verbose=0
)

y_pred_best = np.argmax(best_model.predict(X_test_best, verbose=0), axis=1)
print("\nClassification Report untuk Model Terbaik:")
print(classification_report(y_test, y_pred_best, target_names=le.classes_, zero_division=0))

In [ ]:
# Simpan model terbaik ke file untuk di-upload ke GitHub
model_save_path = f"results/best_model_{best_run_name}.keras"
best_model.save(model_save_path)
print(f"Model terbaik disimpan ke: {model_save_path}")
print(f"\nFile yang perlu di-push ke GitHub:")
print(f"  {model_save_path}")
print(f"  results/experiment_logs.csv")
print(f"  modeling_experiment.ipynb")

In [ ]:
# Jalankan MLflow UI di background
print("MLflow UI berjalan di background.")
print("Akses di: http://127.0.0.1:5050")
get_ipython().system_raw('mlflow ui --port 5050 &')